# Graph-BROS Evaluation: Direction-Agnostic $\beta$-Skeleton Graph Overlay + LoRA

This notebook executes the **Fast-Track Matched-Budget Protocol** (`2 folds x 2 seeds = 4 runs per regime` across Dual T4 GPUs) for **Graph-BROS**:
- **Phase 1 (Visual Phrase Merging & Scale-Invariant Dimensions):** Merges horizontally collinear OCR word fragments (`merge_collinear_boxes`) before computing graph edges.
- **Phase 2 (Direction-Agnostic $\beta$-Skeleton Graph):** Computes 3-D isotropic edge features $\mathbf{e}_{ij} = [\log(d_{ij}+1), \log(\text{Area}_i/\text{Area}_j), \text{IoU}_{\text{axial}}(i,j)]$ and prunes transitive edges via Gabriel Graph ($\beta = 1.0$) exclusion disks.
- **Phase 3 (Graph-BROS Self-Attention):** Injects $\mathbf{W}_g \cdot \mathbf{e}_{ij} + \gamma_h \log(A_{ij} + \epsilon)$ directly into `BrosSelfAttention` pre-softmax scores.
- **Phase 4 (LoRA + Dual-T4 Engineering):** Freezes the 110M base BROS weights and applies Rank-8 LoRA ($r=8, \alpha=16$) to Query/Key projections (~301K trainable parameters, **0.27%** of total weights), with `--no-save-weights` to prevent Kaggle 20GB disk exhaustion.
- **Phase 5 (Focal Loss):** Uses multi-class Focal Loss ($\gamma=2.0$) to handle token class imbalance.

In [ ]:
# 1. Clone / Update Repository and Install Dependencies
import os
import subprocess

REPO_URL = "https://github.com/Biswadeep2609/KeyValue-Extraction.git"
BRANCH = "feat/bros-relative-layout"
WORK_DIR = "/kaggle/working/KeyValue-Extraction"

if not os.path.exists(WORK_DIR):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, WORK_DIR], check=True)
else:
    subprocess.run(["git", "-C", WORK_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", WORK_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", WORK_DIR, "pull", "origin", BRANCH], check=True)

os.chdir(WORK_DIR)
print(f"Working directory: {os.getcwd()}")

subprocess.run(["pip", "install", "-q", "seqeval", "transformers", "datasets", "accelerate", "pyyaml", "scipy"], check=True)
print("Dependencies installed successfully.")

In [ ]:
# 2. Automatic VRDU Dataset Setup
import os
import subprocess
from pathlib import Path

vrdu_target = Path("data/vrdu/registration-form")
if not vrdu_target.exists():
    print("[*] Cloning VRDU dataset repository...")
    os.makedirs("data/vrdu", exist_ok=True)
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/google-research-datasets/vrdu.git", "/tmp/vrdu_repo"], check=True)
    subprocess.run(["cp", "-r", "/tmp/vrdu_repo/registration-form", "data/vrdu/"], check=True)
    if os.path.exists("/tmp/vrdu_repo/ad-buy-form"):
        subprocess.run(["cp", "-r", "/tmp/vrdu_repo/ad-buy-form", "data/vrdu/"], check=True)
    # Decompress dataset.jsonl.gz if needed
    for gz in Path("data/vrdu").rglob("dataset.jsonl.gz"):
        jsonl = gz.with_suffix("")
        if not jsonl.exists():
            subprocess.run(["gunzip", "-k", str(gz)], check=True)
    print("[+] VRDU dataset ready at data/vrdu/registration-form")
else:
    print("[+] VRDU dataset already present.")

In [ ]:
# 3. Pre-Flight Verification: Unit Tests & 1-Step End-to-End Smoke Test
import subprocess
import sys

print("Running Graph-BROS unit tests...")
res_unit = subprocess.run([sys.executable, "tests/test_graph_bros.py"], check=False)
if res_unit.returncode != 0:
    raise RuntimeError("Unit tests failed! Aborting before GPU training.")

print("\nRunning 1-step forward/backward smoke test on configs/vrdu_graph_bros.yaml...")
res_smoke = subprocess.run(
    [sys.executable, "src/smoke_test.py", "--configs", "configs/vrdu_graph_bros.yaml"],
    check=False,
)
if res_smoke.returncode != 0:
    raise RuntimeError("Smoke test failed! Aborting before GPU training.")

In [ ]:
# 4. Run Experiment 5: Graph-BROS (Fast-Track Matched Protocol: 2 Folds x 2 Seeds across Dual T4 GPUs)
import subprocess
import sys

cmd = [
    sys.executable,
    "-u",
    "src/train.py",
    "--config",
    "configs/vrdu_graph_bros.yaml",
    "--protocol",
    "matched",
    "--no-save-weights",
]
print("Launching:", " ".join(cmd))
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f"Graph-BROS training exited with code {proc.returncode}")

In [ ]:
# 5. Master 7-Architecture Cross-Template Generalization Benchmark Table
import json
from pathlib import Path
import pandas as pd

experiments = [
    ("LiLT Baseline (512 Truncated)", "results/vrdu_lilt/metrics.json"),
    ("LiLT + Track 1 (Window + Jitter)", "results/vrdu_track1_lilt/metrics.json"),
    ("Exp 1: BROS Baseline (512 Truncated)", "results/vrdu_bros/metrics.json"),
    ("Exp 2: BROS + Track 1 (Window + Jitter)", "results/vrdu_track1_bros/metrics.json"),
    ("Exp 3: DANN-LiLT (Adversarial + Track 1)", "results/vrdu_dann_lilt/metrics.json"),
    ("Exp 4: DANN-BROS (Adversarial + Track 1)", "results/vrdu_dann_bros/metrics.json"),
    ("Exp 5: Graph-BROS (Beta-Skeleton + LoRA + Track 1)", "results/vrdu_graph_bros/metrics.json"),
]

rows = []
for name, rel_path in experiments:
    p = Path(rel_path)
    if not p.exists():
        continue
    data = json.loads(p.read_text(encoding="utf-8"))
    gap = data.get("gap", {})
    rows.append({
        "Architecture": name,
        "MTL F1 (%)": f"{gap.get('mtl_mean_f1', 0)*100:.2f} +/- {gap.get('mtl_std', 0)*100:.2f}",
        "UTL F1 (%)": f"{gap.get('utl_mean_f1', 0)*100:.2f} +/- {gap.get('utl_std', 0)*100:.2f}",
        "Gap (MTL-UTL)": f"{gap.get('gap_f1', 0)*100:.2f}",
        "Rel. Drop (%)": f"{gap.get('relative_drop_pct', 0):.1f}%",
        "Runs (MTL/UTL)": f"{len(data.get('MTL', []))}/{len(data.get('UTL', []))}",
    })

df = pd.DataFrame(rows)
print("\n" + "=" * 95)
print("MASTER ARCHITECTURAL BENCHMARK: VRDU REGISTRATION-FORM (MATCHED BUDGET N=200)")
print("=" * 95)
print(df.to_string(index=False))
df.to_csv("results/master_architectural_benchmark_with_graph_bros.csv", index=False)